In [2]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

files = reader.read()

In [3]:
documents = []

for file in files:
    doc = file.parse()
    documents.append(doc)

In [4]:
len(documents)

72

In [5]:
documents[0]

{'content': '# Introduction\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn this module, we\'ll build a working Retrieval-Augmented\nGeneration (RAG) system from scratch, step by step.\n\nWe write everything in plain Python. We build a small search index by\nhand and call the LLM ourselves. I want you to see every piece first.\nThat way you know what a framework does for you before you reach for\none.\n\nPlaces where you can find me:\n\n- [My substack](https://alexeyondata.substack.com/)\n- [LinkedIn](https://www.linkedin.com/in/agrigorev/)\n- [X](https://x.com/Al_Grigor)\n\n## LLMs\n\nAn LLM (Large Language Model) is a neural network trained on massive\namounts of text. Given a prompt, it generates a continuation - a\nplausible next piece of text.\n\nThink of your phone. When you type "how are" in WhatsApp, it suggests\n"you" as the next word. "How are you" is the most common continuation.\nYour phone uses a simp

**Q1:** **72** lesson page

In [6]:
from minsearch import Index

index = Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)

index.fit(documents)

In [7]:
question = "How does the agentic loop keep calling the model until it stops?"

search_results = index.search(
    question,
    boost_dict={"question": 2.0, "section": 0.5},
 #   filter_dict={"course": "llm-zoomcamp"},
    num_results=5
)

search_results

[{'content': '# The Agentic Loop\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=ePlQUcTPPjw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn the previous lesson, we did function calling by hand. We sent a\nmessage and got back a function call. We ran it, sent the result back,\nand got the answer.\n\nThat works for one function call. It breaks down when the model wants\nto search several times, or when the first search misses the answer.\nWe don\'t know in advance how many calls the model will want. So we\nneed a loop that keeps calling the model and running tools until it\'s\ndone. An agent is exactly that.\n\n## Anatomy of an agent\n\nWith the LLM in the driver\'s seat, we have an agent. It\'s an AI\nassistant whose goal is to help the user.\n\nAn agent has three parts:\n\n- Instructions, the role and behavior we want. We pass this as the\n  `developer` message. The better the instructions, the better the\n  agent helps.\n- Tools, the functions the agent can call to carry

**Q2:** 'filename': '01-agentic-rag/lessons/14-agentic-loop.md'

In [8]:
from dotenv import load_dotenv
load_dotenv()
from openai import OpenAI
openai_client = OpenAI()

In [9]:
from rag_helper_HW import RAGBase

In [10]:
#documents = load_faq_data()
#index = build_index(documents)

assistant = RAGBase(index, openai_client)

In [11]:
response = assistant.rag("How does the agentic loop keep calling the model until it stops?")

In [12]:
print(response)

It keeps calling the model inside a `while True` loop, and after each response it checks whether the model returned any `function_call` items.

- If there is a function call, the code runs the tool, appends the tool result to `messages`, and loops again.
- If there are no function calls in that turn, it breaks out of the loop and stops.

So the stopping condition is: **no function calls this turn means the model is done**.


In [13]:
usage = assistant.get_last_usage()
usage.input_tokens, usage.output_tokens

(7146, 100)

**Q3:** About 7000

In [14]:
from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)

In [15]:
len(chunks)

295

**Q4:** 295 Chunks

In [16]:
index = Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)

index.fit(chunks)

In [17]:
assistant2 = RAGBase(index, openai_client)

In [18]:
response = assistant2.rag("How does the agentic loop keep calling the model until it stops?")

In [19]:
print(response)

It keeps calling the model inside a `while True` loop. After each model response, the code checks whether any `function_call` items were returned:

- if there is at least one function call, it runs the tool, appends the tool result to the message history, and loops again
- if there are no function calls in that turn, it breaks out of the loop

So the stop condition is: **no function calls this turn means we’re done**.


In [20]:
usage = assistant2.get_last_usage()
usage.input_tokens, usage.output_tokens

(2329, 100)

**Q5:** About 3x fewer

In [21]:
import requests
from minsearch import AppendableIndex
from typing import List, Dict, Any
from toyaikit.chat import IPythonChatInterface, OpenAIClient
from toyaikit.tools import Tools
from toyaikit.chat.runners import OpenAIResponsesRunner

In [22]:
class SearchTools:
    def __init__(self, index):
        self.index = index
        self.search_count = 0
    def search(self, query: str) -> List[Dict[str, Any]]:
        """
        Search the course documents for entries matching the given query.
        
        Args:
            query (str): Search query text to look up in the course documents.
        
        Returns:
            List[Dict[str, Any]]: A list of search result entries.
        """
        boost = {'content': 2.0, 'filename': 0.5}
        results = self.index.search(
            query=query,
            boost_dict=boost,
            num_results=5,
        )
        self.search_count += 1
        return results

In [23]:
search_tools = SearchTools(index)

tools = Tools()
tools.add_tools(search_tools)

developer_prompt = """
You're a course teaching assistant. Answer the student's question using the search tool. 
Make multiple searches with different keywords before answering.
"""


In [24]:
runner = OpenAIResponsesRunner(
    tools=tools,
    developer_prompt=developer_prompt,
    chat_interface=IPythonChatInterface(),
    llm_client=OpenAIClient() 
)

runner.run()

-> Response received


-> Response received


Chat ended.


LoopResult(new_messages=[EasyInputMessage(content="\nYou're a course teaching assistant. Answer the student's question using the search tool. \nMake multiple searches with different keywords before answering.\n", role='developer', phase=None, type=None), EasyInputMessage(content='How does the agentic loop work, and how is it different from plain RAG?', role='user', phase=None, type=None), ResponseFunctionToolCall(arguments='{"query":"agentic loop"}', call_id='call_pdncZR6o8Jjh3G6vw4jJFGum', name='search', type='function_call', id='fc_0947d142c97409f8006a3827b1db7c81a2a362020aa9f6846e', namespace=None, status='completed'), ResponseFunctionToolCall(arguments='{"query":"RAG"}', call_id='call_0oyRdHDaZGZAFbrhIu08yBir', name='search', type='function_call', id='fc_0947d142c97409f8006a3827b1db8881a2bbae1a7cb005155b', namespace=None, status='completed'), {'type': 'function_call_output', 'call_id': 'call_pdncZR6o8Jjh3G6vw4jJFGum', 'output': '[\n  {\n    "start": 4000,\n    "content": "esult.\\n

In [26]:
print("\n" + "=" * 60)
print(f"📊 Total search tool calls: {search_tools.search_count}")
print("=" * 60)


📊 Total search tool calls: 2


**Q6:** About 4 times